In [ ]:
import os
import re
import warnings
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore")


# ---------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------

BASE_DIR = os.path.dirname(os.path.abspath(_file_))
DATA_DIR = os.path.join(BASE_DIR, "data")
OUTPUT_DIR = os.path.join(BASE_DIR, "results", "clustering")

TARGET_COL = "abandono_hash"
RANDOM_STATE = 42

# Sampling is used because the dataset is very large.
# K-Means + Silhouette can be slow on 464k rows.
SAMPLE_SIZE = 30000

# Agglomerative Clustering is much slower, so we use a smaller sample.
AGG_SAMPLE_SIZE = 5000

# Cluster numbers to test
K_RANGE = range(2, 9)

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ID / leakage columns that should not be used for clustering
DROP_COLS = [
    "dni_hash",
    "tit_hash",
    "asi_hash",
    "grupos_por_tipocredito_hash",
    "baja_fecha",
    "fecha_datos",
    "caca",
]


# Monthly LMS / Wi-Fi activity prefixes
LMS_PREFIXES = [
    "pft_events",
    "pft_days_logged",
    "pft_visits",
    "pft_assignment_submissions",
    "pft_test_submissions",
    "pft_total_minutes",
    "n_wifi_days",
    "resource_events",
    "n_resource_days",
]


# Pattern for leftover monthly columns like something_2022_1
MONTHLY_RE = re.compile(r".+\d{4}\d+$")


# ---------------------------------------------------------
# 1. LOAD DATA
# ---------------------------------------------------------

def load_data():
    """
    Loads all CSV files from the data folder.
    The dataset uses semicolon separators.
    """

    csv_files = sorted([f for f in os.listdir(DATA_DIR) if f.endswith(".csv")])

    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in: {DATA_DIR}")

    frames = []

    for file in csv_files:
        path = os.path.join(DATA_DIR, file)

        df = pd.read_csv(
            path,
            sep=";",
            low_memory=False,
            on_bad_lines="skip"
        )

        print(f"Loaded {file}: {df.shape[0]:,} rows x {df.shape[1]} columns")
        frames.append(df)

    combined = pd.concat(frames, ignore_index=True)

    print(f"\nCombined dataset: {combined.shape[0]:,} rows x {combined.shape[1]} columns")

    return combined


# ---------------------------------------------------------
# 2. CONVERT TEXT NUMBERS TO NUMERIC
# ---------------------------------------------------------

def convert_possible_numeric_columns(df):
    """
    Some columns look numeric but are stored as text.
    Example:
    '7,5' should become 7.5
    '10' should become 10

    This function converts object/text columns into numeric columns
    when most values can be converted successfully.
    """

    for col in df.columns:
        if df[col].dtype == object:
            converted = pd.to_numeric(
                df[col].astype(str).str.replace(",", ".", regex=False),
                errors="coerce"
            )

            original_not_null = df[col].notnull().sum()
            converted_not_null = converted.notnull().sum()

            if original_not_null > 0:
                success_rate = converted_not_null / original_not_null
            else:
                success_rate = 0

            # Convert only if at least 70% values are valid numbers
            if success_rate >= 0.70:
                df[col] = converted

    return df


# ---------------------------------------------------------
# 3. PREPROCESS DATA FOR CLUSTERING
# ---------------------------------------------------------

def preprocess_for_clustering(df):
    """
    Cleans and prepares the dataset for clustering.

    Steps:
    1. Keep abandono_hash only for later evaluation.
    2. Remove target, ID, and leakage columns.
    3. Convert possible numeric columns.
    4. Aggregate monthly LMS/Wi-Fi activity columns.
    5. Handle missing values.
    6. One-hot encode categorical columns.
    7. Scale features.
    """

    print("\nPreprocessing data for clustering...")

    if TARGET_COL not in df.columns:
        raise ValueError(f"Target column '{TARGET_COL}' not found in dataset.")

    # Remove rows where target is missing
    df = df.dropna(subset=[TARGET_COL]).copy()

    # True label only for cluster purity later
    # A = dropout, B = non-dropout
    y_true = (df[TARGET_COL] == "A").astype(int)

    # Drop target and ID/leakage columns
    drop_now = [TARGET_COL] + [c for c in DROP_COLS if c in df.columns]
    df.drop(columns=drop_now, inplace=True)

    print(f"Shape after dropping target/ID columns: {df.shape}")

    # Convert comma-decimal and numeric-looking text columns
    df = convert_possible_numeric_columns(df)

    # -----------------------------------------------------
    # Aggregate monthly LMS / Wi-Fi features
    # This is the fixed part that avoids string + integer error.
    # -----------------------------------------------------

    for prefix in LMS_PREFIXES:
        pattern = re.compile(rf"^{re.escape(prefix)}\d{{4}}\d+$")
        month_cols = [c for c in df.columns if pattern.match(c)]

        if month_cols:
            monthly_numeric = df[month_cols].apply(
                lambda col: pd.to_numeric(
                    col.astype(str).str.replace(",", ".", regex=False),
                    errors="coerce"
                )
            )

            df[f"total_{prefix}"] = monthly_numeric.fillna(0).sum(axis=1)
            df.drop(columns=month_cols, inplace=True)

            print(f"Aggregated {len(month_cols)} columns into total_{prefix}")

    # Drop any leftover monthly columns
    leftover_monthly = [c for c in df.columns if MONTHLY_RE.match(c)]

    if leftover_monthly:
        df.drop(columns=leftover_monthly, inplace=True)
        print(f"Dropped {len(leftover_monthly)} leftover monthly columns")

    # Drop columns with more than 60% missing values
    missing_rate = df.isnull().mean()
    high_missing_cols = missing_rate[missing_rate > 0.60].index.tolist()

    if high_missing_cols:
        df.drop(columns=high_missing_cols, inplace=True)
        print(f"Dropped {len(high_missing_cols)} columns with more than 60% missing values")

    # Separate numeric and categorical columns
    num_cols = df.select_dtypes(include="number").columns.tolist()
    cat_cols = df.select_dtypes(exclude="number").columns.tolist()

    print(f"Numeric columns: {len(num_cols)}")
    print(f"Categorical columns: {len(cat_cols)}")

    # Fill missing numeric values with median
    for col in num_cols:
        median_value = df[col].median()

        if pd.isna(median_value):
            median_value = 0

        df[col] = df[col].fillna(median_value)

    # Fill missing categorical values with mode
    for col in cat_cols:
        mode_value = df[col].mode()

        if not mode_value.empty:
            df[col] = df[col].fillna(mode_value.iloc[0])
        else:
            df[col] = df[col].fillna("Unknown")

    # One-hot encode categorical columns
    if cat_cols:
        df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

    # Make sure all columns are numeric
    df = df.apply(pd.to_numeric, errors="coerce")
    df = df.fillna(0)

    print(f"Features after preprocessing: {df.shape[1]}")

    # Sample the dataset for faster clustering
    if SAMPLE_SIZE and len(df) > SAMPLE_SIZE:
        sample_index = df.sample(
            n=SAMPLE_SIZE,
            random_state=RANDOM_STATE
        ).index

        df = df.loc[sample_index]
        y_true = y_true.loc[sample_index]

        print(f"Sampled {SAMPLE_SIZE:,} rows for clustering")

    # Scale features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df)

    print("Scaling completed.")

    return df, X_scaled, y_true


# ---------------------------------------------------------
# 4. CLUSTER PURITY
# ---------------------------------------------------------

def cluster_purity(y_true, cluster_labels):
    """
    Cluster purity checks how well clusters match the actual dropout labels.

    Higher purity means the clusters are more aligned with true labels.

    Note:
    This is only for evaluation after clustering.
    The labels are not used while training the clustering model.
    """

    data = pd.DataFrame({
        "true_label": y_true,
        "cluster": cluster_labels
    })

    total_correct = 0

    for cluster in data["cluster"].unique():
        cluster_data = data[data["cluster"] == cluster]
        most_common_count = cluster_data["true_label"].value_counts().max()
        total_correct += most_common_count

    purity = total_correct / len(data)

    return purity


# ---------------------------------------------------------
# 5. FIND OPTIMAL K
# ---------------------------------------------------------

def find_optimal_k(X_scaled):
    """
    Tests different k values for K-Means.

    Saves:
    1. optimal_k_results.csv
    2. elbow_plot.png
    3. silhouette_plot.png
    """

    print("\nFinding optimal number of clusters...")

    results = []

    for k in K_RANGE:
        print(f"Testing k={k}...")

        kmeans = KMeans(
            n_clusters=k,
            random_state=RANDOM_STATE,
            n_init=10
        )

        labels = kmeans.fit_predict(X_scaled)

        inertia = kmeans.inertia_
        sil = silhouette_score(X_scaled, labels)
        dbi = davies_bouldin_score(X_scaled, labels)

        results.append({
            "k": k,
            "inertia": inertia,
            "silhouette_score": sil,
            "davies_bouldin_index": dbi
        })

    results_df = pd.DataFrame(results)

    results_path = os.path.join(OUTPUT_DIR, "optimal_k_results.csv")
    results_df.to_csv(results_path, index=False)

    # Elbow plot
    plt.figure(figsize=(7, 5))
    plt.plot(results_df["k"], results_df["inertia"], marker="o")
    plt.xlabel("Number of Clusters (k)")
    plt.ylabel("Inertia")
    plt.title("Elbow Method for K-Means")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "elbow_plot.png"), dpi=120)
    plt.close()

    # Silhouette plot
    plt.figure(figsize=(7, 5))
    plt.plot(results_df["k"], results_df["silhouette_score"], marker="o")
    plt.xlabel("Number of Clusters (k)")
    plt.ylabel("Silhouette Score")
    plt.title("Silhouette Score by Number of Clusters")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "silhouette_plot.png"), dpi=120)
    plt.close()

    best_k = int(
        results_df.sort_values(
            "silhouette_score",
            ascending=False
        ).iloc[0]["k"]
    )

    print("\nOptimal K Results:")
    print(results_df.to_string(index=False))
    print(f"\nBest k based on Silhouette Score: {best_k}")

    return best_k, results_df


# ---------------------------------------------------------
# 6. RUN K-MEANS CLUSTERING
# ---------------------------------------------------------

def run_kmeans(X_scaled, y_true, best_k):
    """
    Runs K-Means using the best k.
    """

    print("\nRunning K-Means clustering...")

    model = KMeans(
        n_clusters=best_k,
        random_state=RANDOM_STATE,
        n_init=10
    )

    labels = model.fit_predict(X_scaled)

    sil = silhouette_score(X_scaled, labels)
    dbi = davies_bouldin_score(X_scaled, labels)
    purity = cluster_purity(y_true, labels)

    metrics = {
        "Model": "K-Means",
        "Clusters": best_k,
        "Silhouette Score": round(sil, 4),
        "Davies-Bouldin Index": round(dbi, 4),
        "Cluster Purity": round(purity, 4)
    }

    print("\nK-Means Metrics:")
    print(metrics)

    return labels, metrics


# ---------------------------------------------------------
# 7. RUN AGGLOMERATIVE CLUSTERING
# ---------------------------------------------------------

def run_agglomerative(X_scaled, y_true, best_k):
    """
    Runs Agglomerative Clustering.

    Because Agglomerative Clustering is slow on large datasets,
    it uses a smaller sample.
    """

    print("\nRunning Agglomerative Clustering...")

    if len(X_scaled) > AGG_SAMPLE_SIZE:
        np.random.seed(RANDOM_STATE)
        sample_idx = np.random.choice(
            len(X_scaled),
            AGG_SAMPLE_SIZE,
            replace=False
        )

        X_agg = X_scaled[sample_idx]
        y_agg = y_true.iloc[sample_idx]

        print(f"Using {AGG_SAMPLE_SIZE:,} rows for Agglomerative Clustering")

    else:
        X_agg = X_scaled
        y_agg = y_true

    model = AgglomerativeClustering(
        n_clusters=best_k,
        linkage="ward"
    )

    labels = model.fit_predict(X_agg)

    sil = silhouette_score(X_agg, labels)
    dbi = davies_bouldin_score(X_agg, labels)
    purity = cluster_purity(y_agg, labels)

    metrics = {
        "Model": "Agglomerative Clustering",
        "Clusters": best_k,
        "Silhouette Score": round(sil, 4),
        "Davies-Bouldin Index": round(dbi, 4),
        "Cluster Purity": round(purity, 4)
    }

    print("\nAgglomerative Clustering Metrics:")
    print(metrics)

    return labels, metrics, X_agg, y_agg


# ---------------------------------------------------------
# 8. CLUSTER PROFILING
# ---------------------------------------------------------

def create_cluster_profile(X_original, y_true, labels, model_name):
    """
    Creates cluster profile tables.

    Saves:
    1. cluster summary
    2. cluster feature means
    3. clustered dataset
    """

    print(f"\nCreating cluster profile for {model_name}...")

    profile_df = X_original.copy()
    profile_df["cluster"] = labels
    profile_df["dropout_label"] = y_true.values

    # Cluster size and dropout rate
    cluster_summary = profile_df.groupby("cluster").agg(
        cluster_size=("cluster", "count"),
        dropout_rate=("dropout_label", "mean")
    ).reset_index()

    cluster_summary["dropout_rate"] = cluster_summary["dropout_rate"].round(4)

    # Average numeric features per cluster
    numeric_cols = profile_df.select_dtypes(include="number").columns.tolist()
    numeric_cols = [c for c in numeric_cols if c not in ["cluster", "dropout_label"]]

    feature_means = profile_df.groupby("cluster")[numeric_cols].mean().round(3)

    safe_name = model_name.lower().replace(" ", "").replace("-", "")

    cluster_summary.to_csv(
        os.path.join(OUTPUT_DIR, f"{safe_name}_cluster_summary.csv"),
        index=False
    )

    feature_means.to_csv(
        os.path.join(OUTPUT_DIR, f"{safe_name}_cluster_feature_means.csv")
    )

    profile_df.to_csv(
        os.path.join(OUTPUT_DIR, f"{safe_name}_clustered_data.csv"),
        index=False
    )

    print("\nCluster Summary:")
    print(cluster_summary.to_string(index=False))

    return cluster_summary, feature_means


# ---------------------------------------------------------
# 9. PCA VISUALIZATION
# ---------------------------------------------------------

def plot_clusters_pca(X_scaled, labels, model_name):
    """
    Creates a 2D PCA plot of clusters.
    """

    print(f"\nCreating PCA plot for {model_name}...")

    pca = PCA(n_components=2, random_state=RANDOM_STATE)
    X_pca = pca.fit_transform(X_scaled)

    plt.figure(figsize=(8, 6))
    plt.scatter(
        X_pca[:, 0],
        X_pca[:, 1],
        c=labels,
        s=10,
        alpha=0.6
    )
    plt.xlabel("PCA Component 1")
    plt.ylabel("PCA Component 2")
    plt.title(f"{model_name} Clusters - PCA View")
    plt.tight_layout()

    safe_name = model_name.lower().replace(" ", "").replace("-", "")

    plt.savefig(
        os.path.join(OUTPUT_DIR, f"{safe_name}_pca_clusters.png"),
        dpi=120
    )

    plt.close()


# ---------------------------------------------------------
# 10. SAVE INTERPRETATION TEMPLATE
# ---------------------------------------------------------

def save_interpretation_notes():
    """
    Saves a text file explaining how to interpret the results.
    """

    notes = """
STUDENT RISK PROFILING - CLUSTERING INTERPRETATION NOTES

1. K-Means Clustering:
K-Means groups students into k clusters based on academic, LMS, and campus activity patterns.

2. Agglomerative Clustering:
Agglomerative clustering is a hierarchical method. It starts with each student as an individual cluster and gradually merges similar students into larger groups.

3. Elbow Method:
The Elbow Method helps identify a good number of clusters by checking where inertia starts decreasing slowly.

4. Silhouette Score:
Silhouette Score measures how well-separated the clusters are.
Higher score is better.
Range: -1 to 1.

5. Davies-Bouldin Index:
Davies-Bouldin Index measures how similar clusters are to each other.
Lower score is better.

6. Cluster Purity:
Cluster Purity compares the unsupervised clusters with actual dropout labels after clustering.
Higher purity means the clusters are more aligned with dropout/non-dropout groups.

7. Dropout Rate:
Dropout rate inside each cluster helps identify risk levels:
- High dropout rate cluster = high-risk student group
- Medium dropout rate cluster = moderate-risk student group
- Low dropout rate cluster = low-risk student group

Important:
The dropout label was not used during clustering.
It was only used after clustering for evaluation and interpretation.
"""

    with open(os.path.join(OUTPUT_DIR, "clustering_interpretation_notes.txt"), "w") as file:
        file.write(notes)


# ---------------------------------------------------------
# 11. MAIN FUNCTION
# ---------------------------------------------------------

def main():
    print("=" * 60)
    print("STUDENT RISK PROFILING - CLUSTERING")
    print("=" * 60)

    print("\n[1] Loading data...")
    df = load_data()

    print("\n[2] Preprocessing data...")
    X_original, X_scaled, y_true = preprocess_for_clustering(df)

    print("\n[3] Finding optimal number of clusters...")
    best_k, optimal_results = find_optimal_k(X_scaled)

    print("\n[4] Running K-Means...")
    kmeans_labels, kmeans_metrics = run_kmeans(X_scaled, y_true, best_k)

    print("\n[5] Profiling K-Means clusters...")
    create_cluster_profile(
        X_original,
        y_true,
        kmeans_labels,
        "K-Means"
    )

    print("\n[6] Creating K-Means PCA plot...")
    plot_clusters_pca(
        X_scaled,
        kmeans_labels,
        "K-Means"
    )

    print("\n[7] Running Agglomerative Clustering...")
    agg_labels, agg_metrics, X_agg, y_agg = run_agglomerative(
        X_scaled,
        y_true,
        best_k
    )

    print("\n[8] Creating Agglomerative PCA plot...")
    plot_clusters_pca(
        X_agg,
        agg_labels,
        "Agglomerative Clustering"
    )

    # Final model comparison
    metrics_df = pd.DataFrame([
        kmeans_metrics,
        agg_metrics
    ])

    metrics_df.to_csv(
        os.path.join(OUTPUT_DIR, "clustering_model_comparison.csv"),
        index=False
    )

    print("\nFinal Clustering Model Comparison:")
    print(metrics_df.to_string(index=False))

    save_interpretation_notes()

    print("\nDone!")
    print(f"All clustering outputs saved to: {OUTPUT_DIR}")


if _name_ == "_main_":
    main()